# RA — Renewable AI Decision Engine (Notebook Demo)

This notebook is a **second front end** for the same RA engine that powers the FastAPI + React web app in `../backend` and `../frontend`. It is not a separate reimplementation — every number produced here comes from the exact same shared logic in [`ra_core/`](../ra_core): the synthetic data generator, the `GradientBoostingRegressor` forecaster, and the decision engine's scoring + explanation logic.

**Why this exists:** the web dashboard is the polished, judge-facing demo. This notebook is for exploring the engine directly — switching scenarios, running "what-if" conditions, and inspecting every intermediate number — without needing two servers running.

Run this top-to-bottom (`Run All`) and Section 2 will show the exact same current-state numbers as the web dashboard shows on a freshly-seeded `sunny` scenario (same seed, same default time step).

**Part 1 note:** `ra_core` now supports multiple synthetic stations (see `ra_core/stations.py`). This notebook intentionally keeps using a single station -- the same default (`DEFAULT_STATION_ID = "hybrid-01"`) the web app falls back to when no station is specified -- so it stays simple. A multi-station notebook view is out of scope for this part.


## 1. Setup

Locate and import `ra_core`, then generate the same deterministic synthetic dataset the web app uses (one full multi-day trace per scenario).

In [ ]:
import sys
from pathlib import Path


def _find_repo_root(start: Path, marker: str = "ra_core", max_up: int = 5) -> Path:
    p = start.resolve()
    for _ in range(max_up + 1):
        if (p / marker).is_dir():
            return p
        p = p.parent
    raise RuntimeError(
        f"Could not locate '{marker}/' above {start} -- launch Jupyter from the repo root "
        "(the folder containing ra_core/, backend/, frontend/, notebook/)."
    )


_repo_root = _find_repo_root(Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

print(f"ra_core loaded from: {_repo_root / 'ra_core'}")


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output

from ra_core.config import (
    SCENARIOS, DEFAULT_SCENARIO, DEFAULT_START_INDEX, TOTAL_POINTS,
)
from ra_core.data_generator import generate_series
from ra_core.forecasting import forecast_surplus
from ra_core.decision_engine import evaluate, status_from_priority, STATUS_LABELS
from ra_core.stations import DEFAULT_STATION_ID, get_station, list_stations
from ra_core.what_if import simulate_what_if, WhatIfValidationError
from ra_core import kpi

# RA now supports multiple synthetic stations (Part 1). This notebook keeps
# using the single default station (DEFAULT_STATION_ID = "hybrid-01") --
# no multi-station notebook UI is added here, only an explicit reference to
# the same default the web app falls back to when station_id is omitted.
station = get_station(DEFAULT_STATION_ID)

# One full deterministic trace per scenario for the default station --
# identical to what the web app seeds into SQLite on first startup (same
# seed, same generator, same station).
state = {
    "dfs": {s: generate_series(s, station_id=DEFAULT_STATION_ID) for s in SCENARIOS},
    "decision_log": [],       # notebook-session log ("Log this Decision" below)
    "last_result": None,
    "last_reading": None,
    "last_scenario": DEFAULT_SCENARIO,
    "last_index": DEFAULT_START_INDEX,
    "last_forecast": None,
}

print(f"Station: {station.name} ({station.id}, {station.energy_type})")
print(f"Generated {len(SCENARIOS)} scenarios x {TOTAL_POINTS} timesteps "
      f"({TOTAL_POINTS * 15 / 60 / 24:.0f} days at 15-min resolution).")


### Display helpers

Small HTML/Plotly formatting helpers so the engine's output reads as cards and tables instead of raw dicts. These format numbers only -- they don't compute anything; every number they display comes straight out of `ra_core`.

In [ ]:
ACTION_LABELS = {
    "battery_charge": "Battery Charge",
    "battery_discharge": "Battery Discharge",
    "water_pumping": "Water Pumping / Desalination",
    "sell_grid": "Sell to Grid",
    "grid_import": "Grid Support",
    "curtail": "Curtailment",
}
ACTION_COLORS = {
    "battery_charge": "#059669",
    "battery_discharge": "#7c3aed",
    "water_pumping": "#0891b2",
    "sell_grid": "#d97706",
    "grid_import": "#e11d48",
    "curtail": "#64748b",
}
MODE_COLORS = {"surplus": "#059669", "deficit": "#e11d48"}
PRIORITY_COLORS = {"critical": "#e11d48", "high": "#d97706", "medium": "#0284c7", "normal": "#64748b"}


def _card(label, value, sub=""):
    sub_html = f'<div style="font-size:11px;color:#94a3b8;">{sub}</div>' if sub else ""
    return (
        '<div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;'
        'padding:12px 16px;min-width:130px;">'
        f'<div style="font-size:11px;letter-spacing:.05em;text-transform:uppercase;color:#64748b;">{label}</div>'
        f'<div style="font-size:20px;font-weight:600;color:#0f172a;">{value}</div>{sub_html}</div>'
    )


def current_state_html(reading, surplus_kw):
    gen = reading["solar_kw"] + reading["wind_kw"]
    label = "Surplus" if surplus_kw >= 0 else "Deficit"
    color = "#059669" if surplus_kw >= 0 else "#dc2626"
    surplus_card = (
        f'<div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;padding:12px 16px;min-width:130px;">'
        f'<div style="font-size:11px;letter-spacing:.05em;text-transform:uppercase;color:#64748b;">{label}</div>'
        f'<div style="font-size:20px;font-weight:600;color:{color};">{abs(surplus_kw):.1f} kW</div></div>'
    )
    cards = [
        _card("Generation", f"{gen:.1f} kW", f"Solar {reading['solar_kw']:.1f} · Wind {reading['wind_kw']:.1f}"),
        _card("Demand", f"{reading['demand_kw']:.1f} kW"),
        surplus_card,
        _card("Battery SoC", f"{reading['battery_soc']:.0f}%"),
        _card("Grid Price", f"{reading['price_egp']:.2f} EGP/kWh"),
        _card("Cloud Cover", f"{reading['cloud_cover'] * 100:.0f}%"),
        _card("Wind Speed", f"{reading['wind_speed']:.1f} m/s"),
        _card("Simulated Time", reading["timestamp"].replace("T", " ")[:16]),
    ]
    return f'<div style="display:flex;flex-wrap:wrap;gap:10px;margin:8px 0 16px;">{"".join(cards)}</div>'


def decision_card_html(result):
    """result: the FULL evaluate() output (not just result["recommended"]) --
    Part 3 needs mode/priority/before/after/remaining-deficit, which live at
    the top level of the result, alongside the recommended action itself.
    """
    rec = result["recommended"]
    mode, priority = result["mode"], result["priority"]
    before, after = result["before"], result["after"]
    remaining = result["remaining_deficit_kw"]
    secondary, secondary_amount = result["secondary_action"], result["secondary_amount_kw"]

    label = ACTION_LABELS.get(rec["action"], rec["action"])
    mode_color = MODE_COLORS.get(mode, "#64748b")
    priority_color = PRIORITY_COLORS.get(priority, "#64748b")

    is_grid_import = rec["action"] == "grid_import"
    money_label = "Expected Cost" if is_grid_import else "Expected Value"
    money_value = rec["expected_cost_egp"] if is_grid_import else rec["expected_value_egp"]
    co2_label = "CO2 Emitted" if is_grid_import else "CO2 Avoided"
    co2_value = rec["co2_emitted_kg"] if is_grid_import else rec["co2_avoided_kg"]
    money_color = "#e11d48" if is_grid_import else "#059669"
    co2_color = "#e11d48" if is_grid_import else "#0284c7"

    remaining_html = ""
    if remaining and remaining > 0.05:
        secondary_label = ACTION_LABELS.get(secondary, secondary) if secondary else ""
        extra = f' — {secondary_label} needed for {secondary_amount:.1f} kW' if secondary else ''
        remaining_html = (
            '<div style="background:#fef2f2;border:1px solid #fecaca;border-radius:8px;'
            'padding:8px 12px;font-size:12.5px;color:#991b1b;margin-bottom:10px;">'
            f'Remaining deficit: <b>{remaining:.1f} kW</b>{extra}</div>'
        )

    reason_html = (
        f'<div style="font-size:12px;color:#94a3b8;margin-bottom:6px;">{rec["reason"]}</div>'
        if rec.get("reason") else ""
    )

    return f'''
    <div style="border:1px solid #e2e8f0;border-radius:12px;padding:16px 20px;background:#ffffff;">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
        <div style="font-weight:700;font-size:15px;color:#0f172a;">Recommended: {label}</div>
        <div style="display:flex;gap:6px;">
          <span style="background:{mode_color};color:white;font-size:11px;padding:3px 9px;border-radius:999px;text-transform:capitalize;">{mode}</span>
          <span style="background:{priority_color};color:white;font-size:11px;padding:3px 9px;border-radius:999px;text-transform:capitalize;">{priority}</span>
        </div>
      </div>
      <div style="display:flex;gap:16px;margin:10px 0;flex-wrap:wrap;font-size:12px;color:#475569;">
        <div>Before: gen {before["generation_kw"]:.1f} kW · demand {before["demand_kw"]:.1f} kW · net {before["net_balance_kw"]:.1f} kW · battery {before["battery_soc_pct"]:.0f}%</div>
        <div>After: net {after["net_balance_kw"]:.1f} kW · battery {after["battery_soc_pct"]:.0f}%</div>
      </div>
      {remaining_html}
      {reason_html}
      <div style="display:flex;gap:24px;margin:14px 0;flex-wrap:wrap;">
        <div><div style="font-size:18px;font-weight:600;">{rec['amount_kw']:.1f} kW</div><div style="font-size:11px;color:#64748b;">Amount</div></div>
        <div><div style="font-size:18px;font-weight:600;color:{money_color};">{money_value:.1f} EGP</div><div style="font-size:11px;color:#64748b;">{money_label}</div></div>
        <div><div style="font-size:18px;font-weight:600;color:{co2_color};">{co2_value:.1f} kg</div><div style="font-size:11px;color:#64748b;">{co2_label}</div></div>
        <div><div style="font-size:18px;font-weight:600;color:#7c3aed;">{rec['score']:.1f}</div><div style="font-size:11px;color:#64748b;">Decision score*</div></div>
      </div>
      <div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;padding:12px 14px;font-size:13.5px;line-height:1.5;color:#334155;">{rec['explanation']}</div>
      <div style="font-size:10.5px;color:#94a3b8;margin-top:6px;">*Score = expected value (minus cost, for grid support) + a small CO2 weighting used to rank actions. It is a transparent value ranking, not a probabilistic confidence.</div>
    </div>'''


def ranked_actions_html(ranked):
    def _row(a):
        is_grid_import = a["action"] == "grid_import"
        money = -a["expected_cost_egp"] if is_grid_import else a["expected_value_egp"]
        money_color = "#e11d48" if money < 0 else "#059669"
        return (
            f'<tr><td style="padding:6px 10px;">{ACTION_LABELS.get(a["action"], a["action"])}</td>'
            f'<td style="padding:6px 10px;text-align:right;">{a["amount_kw"]:.1f} kW</td>'
            f'<td style="padding:6px 10px;text-align:right;color:{money_color};">{money:.1f} EGP</td>'
            f'<td style="padding:6px 10px;text-align:right;font-weight:600;">{a["score"]:.1f}</td></tr>'
        )
    rows = "".join(_row(a) for a in ranked)
    return f'''
    <table style="border-collapse:collapse;width:100%;font-size:13.5px;">
      <thead><tr style="border-bottom:2px solid #e2e8f0;color:#64748b;text-align:right;">
        <th style="text-align:left;padding:6px 10px;">Action</th><th>Amount</th><th>Value</th><th>Score</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>'''


def build_forecast_figure(fc):
    """Component forecast chart: separate solar/wind/demand lines (Part 2),
    each with a shaded empirical uncertainty band for the forecast portion.
    A structurally zero-capacity source (solar_method/wind_method ==
    "structural_zero") is omitted from the chart rather than drawn as a
    flat, misleading zero line.
    """
    history = fc["history"]
    future = fc["forecast"]
    h_times = [h["timestamp"] for h in history]
    f_times = [f["timestamp"] for f in future]
    now_time = h_times[-1] if h_times else (f_times[0] if f_times else None)

    solar_available = not future or future[0]["solar_method"] != "structural_zero"
    wind_available = not future or future[0]["wind_method"] != "structural_zero"

    fig = go.Figure()

    def add_component(key, color, available):
        if not available:
            return
        h_actual = [h[f"actual_{key}_kw"] for h in history]
        f_pred = [pt[f"{key}_kw"] for pt in future]
        f_lower = [pt[f"{key}_lower_kw"] for pt in future]
        f_upper = [pt[f"{key}_upper_kw"] for pt in future]
        bridge_val = h_actual[-1] if h_actual else (f_pred[0] if f_pred else None)

        fig.add_trace(go.Scatter(
            x=h_times, y=h_actual, name=f"{key.capitalize()} (actual)",
            mode="lines", line=dict(color=color, width=2), legendgroup=key,
        ))
        if f_times:
            fig.add_trace(go.Scatter(
                x=[now_time] + f_times, y=[bridge_val] + f_pred,
                name=f"{key.capitalize()} (forecast)", mode="lines",
                line=dict(color=color, width=2, dash="dash"), legendgroup=key,
            ))
            # Shaded empirical uncertainty band (~80% nominal coverage).
            fig.add_trace(go.Scatter(
                x=f_times + f_times[::-1], y=f_upper + f_lower[::-1],
                fill="toself", fillcolor=color, opacity=0.15,
                line=dict(width=0), hoverinfo="skip", showlegend=False,
                legendgroup=key,
            ))

    add_component("solar", "#f59e0b", solar_available)
    add_component("wind", "#0ea5e9", wind_available)
    add_component("demand", "#8b5cf6", True)

    if now_time:
        # Use add_shape + add_annotation instead of the add_vline/add_hline
        # convenience wrappers -- those have known cross-version kwarg
        # forwarding issues (TypeError deep inside BaseFigure.add_vline)
        # depending on the installed plotly version. add_shape/add_annotation
        # are the stable, low-level primitives and avoid that failure mode.
        fig.add_shape(
            type="line", xref="x", yref="paper",
            x0=now_time, x1=now_time, y0=0, y1=1,
            line=dict(color="#f59e0b", width=1, dash="dot"),
        )
        fig.add_annotation(
            x=now_time, y=1.0, yref="paper", yanchor="bottom",
            text="now", showarrow=False, font=dict(color="#f59e0b", size=12),
        )
    fig.update_layout(
        height=360, margin=dict(l=40, r=20, t=30, b=40),
        yaxis_title="kW", template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    return fig


def confidence_summary_html(future):
    """Compact next-hour (T+1h, ~4 steps @ 15min) confidence readout --
    mirrors the web dashboard's "Next-Hour Forecast Confidence" strip.
    Confidence is a model-confidence score (derived from empirical
    residual-quantile interval width), not a probability of correctness.
    """
    if not future:
        return '<div style="color:#94a3b8;font-size:12px;">No forecast available.</div>'
    point = future[3] if len(future) > 3 else future[-1]

    def pill(label, method_key, conf_key):
        if method_key and point.get(method_key) == "structural_zero":
            value = "n/a"
        else:
            v = point.get(conf_key)
            value = f"{v:.0f}%" if v is not None else "n/a"
        return (
            f'<div style="text-align:center;padding:0 10px;">'
            f'<div style="font-size:10px;letter-spacing:.05em;text-transform:uppercase;color:#64748b;">{label}</div>'
            f'<div style="font-size:14px;font-weight:600;color:#0f172a;">{value}</div></div>'
        )

    cells_html = (
        pill("Solar", "solar_method", "solar_confidence_pct")
        + pill("Wind", "wind_method", "wind_confidence_pct")
        + pill("Demand", None, "demand_confidence_pct")
        + pill("Net Balance", None, "net_balance_confidence_pct")
    )
    return (
        '<div style="font-size:11px;color:#94a3b8;margin-bottom:4px;">'
        "Next-Hour Forecast Confidence (model-confidence score, not a probability)</div>"
        f'<div style="display:flex;">{cells_html}</div>'
    )


## 2–5. Live State → Forecast → Decision → Ranked Actions

These four views are wired together, exactly like the web dashboard: move the **Time step** slider (or click **+15 min** / **+1 hour**) to advance the simulated clock, or switch **Scenario**, and all four sections below update together. Click **Log this Decision** to append the current recommendation to the Section 8 history log.

In [ ]:
scenario_dd = widgets.Dropdown(options=SCENARIOS, value=DEFAULT_SCENARIO, description="Scenario:")
index_slider = widgets.IntSlider(value=DEFAULT_START_INDEX, min=0, max=TOTAL_POINTS - 1, step=1,
                                  description="Time step:", continuous_update=False,
                                  layout=widgets.Layout(width="520px"))
advance_15 = widgets.Button(description="+15 min")
advance_1h = widgets.Button(description="+1 hour")
log_button = widgets.Button(description="Log this Decision", button_style="primary")
log_feedback = widgets.HTML("")


def _on_scenario_change(change):
    if change["name"] == "value":
        index_slider.value = DEFAULT_START_INDEX


scenario_dd.observe(_on_scenario_change, names="value")
advance_15.on_click(lambda _: setattr(index_slider, "value", min(index_slider.value + 1, TOTAL_POINTS - 1)))
advance_1h.on_click(lambda _: setattr(index_slider, "value", min(index_slider.value + 4, TOTAL_POINTS - 1)))


def render_dashboard(scenario, idx):
    df = state["dfs"][scenario]
    reading = df.iloc[idx].to_dict()
    all_rows = df.to_dict("records")
    future_rows = df.iloc[idx + 1: idx + 13].to_dict("records")
    fc = forecast_surplus(all_rows, idx)
    future_prices = [r["price_egp"] for r in future_rows]
    surplus_kw = reading["solar_kw"] + reading["wind_kw"] - reading["demand_kw"]
    # Pass the default station's actual battery configuration (Part 3) --
    # not just its capacity -- so charge/discharge rate limits, min/max SoC,
    # and efficiencies match what the web app uses for the same station.
    result = evaluate(
        reading, fc["forecast"], future_prices,
        battery_capacity_kwh=station.battery_capacity_kwh,
        battery_charge_limit_kw=station.battery_charge_limit_kw,
        battery_discharge_limit_kw=station.battery_discharge_limit_kw,
        battery_min_soc_pct=station.battery_min_soc_pct,
        battery_max_soc_pct=station.battery_max_soc_pct,
        battery_charge_efficiency=station.battery_charge_efficiency,
        battery_discharge_efficiency=station.battery_discharge_efficiency,
    )

    state.update(last_result=result, last_reading=reading, last_scenario=scenario,
                 last_index=idx, last_forecast=fc)

    display(Markdown("#### 2. Current State"))
    display(HTML(current_state_html(reading, surplus_kw)))

    display(Markdown("#### 3. Forecast (next 6h)"))
    display(build_forecast_figure(fc))
    mq = fc["model_quality"]
    solar_mae = mq["solar_mae_kw"] if mq["solar_mae_kw"] is not None else "n/a"
    wind_mae = mq["wind_mae_kw"] if mq["wind_mae_kw"] is not None else "n/a"
    display(HTML(
        f'<div style="font-size:12px;color:#64748b;margin:-8px 0 8px;">'
        f'Validation MAE ({mq["validation_method"]}) — solar {solar_mae} kW · wind {wind_mae} kW '
        f'· demand {mq["demand_mae_kw"]} kW · generation {mq["generation_mae_kw"]} kW '
        f'· net balance {mq["net_balance_mae_kw"]} kW</div>'
    ))
    display(HTML(confidence_summary_html(fc["forecast"])))

    display(Markdown("#### 4. AI Decision Engine"))
    display(HTML(decision_card_html(result)))

    display(Markdown("#### 5. Priority Queue — Ranked Actions"))
    display(HTML(ranked_actions_html(result["ranked_actions"])))


def _on_log(_):
    if state["last_result"] is None:
        return
    result = state["last_result"]
    # Flatten the top-level decision metadata onto the recommended action --
    # same convention backend/app/main.py uses for /decision/log, so the KPI
    # section below (which reads mode/action/expected_cost_egp/etc.) works
    # identically whether the decision came from the API or the notebook.
    rec = {
        **result["recommended"],
        "timestamp": state["last_reading"]["timestamp"],
        "scenario": state["last_scenario"],
        "mode": result["mode"],
        "priority": result["priority"],
    }
    state["decision_log"].append(rec)
    log_feedback.value = (
        f'<span style="color:#059669;font-size:12px;">Logged decision #{len(state["decision_log"])} '
        f'— {ACTION_LABELS.get(rec["action"], rec["action"])}</span>'
    )


log_button.on_click(_on_log)

controls = widgets.HBox([scenario_dd, index_slider, advance_15, advance_1h, log_button])
dashboard_view = widgets.interactive_output(render_dashboard, {"scenario": scenario_dd, "idx": index_slider})
display(controls, log_feedback, dashboard_view)


## 6. What-If Simulator

Explore hypothetical station assumptions without altering the underlying scenario data. This calls `ra_core.what_if.simulate_what_if()` -- the exact same shared-core function the web dashboard's `POST /simulate` endpoint uses (Part 5) -- so any change you see here comes from the same deterministic pipeline, not a notebook-only calculation.

- **Solar capacity**, **Wind capacity**, **Demand**, and **Battery capacity** are the four supported hypothetical changes (percent), matching the web app exactly.
- Baseline and hypothetical are regenerated fresh from the same scenario/index/random seed -- only the requested capacity/demand assumptions differ, so the comparison isolates the effect of your change.
- This is fully side-effect free: it does not modify `state`, the station registry, or the dashboard above.
- Sliders update the comparison live — no need to re-run the cell.

*Note: an earlier version of this section had its own "dust storm" toggle and directly multiplied historical solar/demand columns. That was notebook-only duplicated logic (no such concept exists in `ra_core`), so Part 5 replaced it with the shared, capacity-driven simulator described above instead of carrying it forward.*

In [ ]:
solar_pct_slider = widgets.FloatSlider(value=0, min=-50, max=100, step=5, description="Solar capacity %")
wind_pct_slider = widgets.FloatSlider(value=0, min=-50, max=100, step=5, description="Wind capacity %")
demand_pct_slider = widgets.FloatSlider(value=0, min=-30, max=50, step=5, description="Demand %")
battery_pct_slider = widgets.FloatSlider(value=0, min=-50, max=100, step=5, description="Battery capacity %")


def _whatif_delta_color(v, higher_is_better=True):
    if not isinstance(v, (int, float)) or abs(v) < 0.05:
        return "#64748b"
    positive = v > 0 if higher_is_better else v < 0
    return "#059669" if positive else "#dc2626"


def _whatif_row(label, base_val, hyp_val, delta_val, higher_is_better=True):
    color = _whatif_delta_color(delta_val, higher_is_better)
    delta_str = f"{delta_val:+.1f}" if isinstance(delta_val, (int, float)) else str(delta_val)
    return (
        f'<tr><td style="padding:4px 8px;color:#64748b;">{label}</td>'
        f'<td style="padding:4px 8px;text-align:right;">{base_val}</td>'
        f'<td style="padding:4px 8px;text-align:right;font-weight:600;">{hyp_val}</td>'
        f'<td style="padding:4px 8px;text-align:right;color:{color};">{delta_str}</td></tr>'
    )


def whatif_comparison_html(result):
    """result: the full simulate_what_if() output. Every number displayed
    here is read straight from result["baseline"]/["hypothetical"]/["impact"]
    -- nothing is recalculated in this display helper."""
    baseline, hyp, impact = result["baseline"], result["hypothetical"], result["impact"]
    action_row_delta = "changed" if impact["decision_changed"] else "same"
    rows = "".join([
        _whatif_row("Generation (1h avg)", f'{baseline["forecast_generation_kw"]:.1f} kW',
                    f'{hyp["forecast_generation_kw"]:.1f} kW', impact["generation_change_kw"]),
        _whatif_row("Demand (1h avg)", f'{baseline["forecast_demand_kw"]:.1f} kW',
                    f'{hyp["forecast_demand_kw"]:.1f} kW', impact["demand_change_kw"], higher_is_better=False),
        _whatif_row("Net Balance (1h avg)", f'{baseline["forecast_net_balance_kw"]:.1f} kW',
                    f'{hyp["forecast_net_balance_kw"]:.1f} kW', impact["net_balance_change_kw"]),
        _whatif_row("Battery Capacity", f'{baseline["battery_capacity_kwh"]:.1f} kWh',
                    f'{hyp["battery_capacity_kwh"]:.1f} kWh', impact["battery_capacity_change_kwh"]),
        _whatif_row("Recommended Action", ACTION_LABELS.get(baseline["recommended_action"], baseline["recommended_action"]),
                    ACTION_LABELS.get(hyp["recommended_action"], hyp["recommended_action"]), action_row_delta),
        _whatif_row("Expected Value", f'{baseline["expected_value_egp"]:.1f} EGP',
                    f'{hyp["expected_value_egp"]:.1f} EGP', impact["expected_value_change_egp"]),
        _whatif_row("Expected Cost", f'{baseline["expected_cost_egp"]:.1f} EGP',
                    f'{hyp["expected_cost_egp"]:.1f} EGP', impact["expected_cost_change_egp"], higher_is_better=False),
        _whatif_row("CO2 Avoided", f'{baseline["co2_avoided_kg"]:.1f} kg',
                    f'{hyp["co2_avoided_kg"]:.1f} kg', impact["co2_avoided_change_kg"]),
        _whatif_row("CO2 Emitted", f'{baseline["co2_emitted_kg"]:.1f} kg',
                    f'{hyp["co2_emitted_kg"]:.1f} kg', impact["co2_emitted_change_kg"], higher_is_better=False),
        _whatif_row("Remaining Deficit", f'{baseline["remaining_deficit_kw"]:.1f} kW',
                    f'{hyp["remaining_deficit_kw"]:.1f} kW', impact["remaining_deficit_change_kw"], higher_is_better=False),
    ])
    return (
        '<table style="width:100%;border-collapse:collapse;font-size:13px;">'
        '<thead><tr style="border-bottom:1px solid #e2e8f0;">'
        '<th style="text-align:left;padding:4px 8px;color:#64748b;">Metric</th>'
        '<th style="text-align:right;padding:4px 8px;color:#64748b;">Baseline</th>'
        '<th style="text-align:right;padding:4px 8px;color:#64748b;">Hypothetical</th>'
        '<th style="text-align:right;padding:4px 8px;color:#64748b;">&Delta; Impact</th>'
        '</tr></thead><tbody>' + rows + '</tbody></table>'
    )


def run_whatif(solar_pct, wind_pct, demand_pct, battery_pct):
    if state["last_result"] is None:
        display(HTML('<div style="color:#94a3b8;">Run Section 2–5 above first.</div>'))
        return

    try:
        result = simulate_what_if(
            station.id, state["last_scenario"], state["last_index"],
            solar_capacity_change_pct=solar_pct,
            wind_capacity_change_pct=wind_pct,
            demand_change_pct=demand_pct,
            battery_capacity_change_pct=battery_pct,
        )
    except WhatIfValidationError as e:
        display(HTML(f'<div style="color:#dc2626;">{e}</div>'))
        return

    display(Markdown(
        "**Baseline vs. What-If recommendation** *(same `ra_core.what_if.simulate_what_if` the web "
        "dashboard\'s `POST /simulate` endpoint calls)*"
    ))
    display(HTML(whatif_comparison_html(result)))
    display(HTML(f'<div style="margin-top:10px;font-size:13px;color:#334155;">{result["explanation"]}</div>'))


whatif_controls = widgets.VBox([
    widgets.HBox([solar_pct_slider, wind_pct_slider]),
    widgets.HBox([demand_pct_slider, battery_pct_slider]),
])
whatif_view = widgets.interactive_output(
    run_whatif,
    {"solar_pct": solar_pct_slider, "wind_pct": wind_pct_slider,
     "demand_pct": demand_pct_slider, "battery_pct": battery_pct_slider},
)
display(whatif_controls, whatif_view)


## 7. RA Assistant (offline, grounded)

A small dropdown demo of the shared `ra_core.assistant` module -- the exact same `classify_intent()` / `answer_question()` functions the web dashboard's `POST /assistant/query` endpoint calls. Pick one of the six supported question types below; RA answers using only the current dashboard state from Section 2-5 above (plus, for the What-If example, a small demo hypothetical run through the same `simulate_what_if()` used in Section 6). This is a demonstration of the shared function, not a notebook chat application -- there is no free-text input or conversation history here.

In [ ]:
from ra_core.assistant import AssistantContext, answer_question

ASSISTANT_EXAMPLES = {
    "explain_current_status": "What is happening now?",
    "explain_forecast": "What is expected during the next six hours?",
    "explain_decision": "Why was this decision selected?",
    "compare_stations": "Which station needs attention?",
    "explain_what_if": "What changed in the latest What-If simulation?",
    "help": "What can I ask?",
}

assistant_dd = widgets.Dropdown(
    options=list(ASSISTANT_EXAMPLES.keys()),
    description="Question type:",
    style={"description_width": "initial"},
)


def _assistant_stations_overview():
    """Small, notebook-local equivalent of GET /stations/overview -- built
    from the exact same shared functions (generate_series/forecast_surplus/
    evaluate/status_from_priority), for the compare_stations example only.
    Not a new calculation."""
    scenario = state["last_scenario"]
    idx = state["last_index"]
    overview = []
    for s in list_stations():
        df_s = generate_series(scenario, station_id=s.id)
        all_rows_s = df_s.to_dict("records")
        reading_s = all_rows_s[idx]
        future_rows_s = all_rows_s[idx + 1: idx + 13]
        fc_s = forecast_surplus(all_rows_s, idx, station_id=s.id)
        future_prices_s = [r["price_egp"] for r in future_rows_s]
        result_s = evaluate(
            reading_s, fc_s["forecast"], future_prices_s,
            battery_capacity_kwh=s.battery_capacity_kwh,
            battery_charge_limit_kw=s.battery_charge_limit_kw,
            battery_discharge_limit_kw=s.battery_discharge_limit_kw,
            battery_min_soc_pct=s.battery_min_soc_pct,
            battery_max_soc_pct=s.battery_max_soc_pct,
            battery_charge_efficiency=s.battery_charge_efficiency,
            battery_discharge_efficiency=s.battery_discharge_efficiency,
        )
        status = status_from_priority(result_s["priority"])
        overview.append({
            "station_id": s.id, "name": s.name, "energy_type": s.energy_type,
            "generation_kw": round(reading_s["solar_kw"] + reading_s["wind_kw"], 2),
            "demand_kw": round(reading_s["demand_kw"], 2),
            "net_balance_kw": result_s["net_balance_kw"],
            "battery_soc_pct": round(reading_s["battery_soc"], 1),
            "mode": result_s["mode"], "priority": result_s["priority"],
            "recommended_action": result_s["recommended"]["action"],
            "status": status, "status_label": STATUS_LABELS.get(status, "Unknown"),
        })
    return overview


def run_assistant(question_type):
    if state["last_result"] is None:
        display(HTML('<div style="color:#94a3b8;">Run Section 2–5 above first.</div>'))
        return

    question = ASSISTANT_EXAMPLES[question_type]
    reading = state["last_reading"]
    current_state = {
        "generation_kw": round(reading["solar_kw"] + reading["wind_kw"], 2),
        "demand_kw": round(reading["demand_kw"], 2),
        "net_balance_kw": round(reading["solar_kw"] + reading["wind_kw"] - reading["demand_kw"], 2),
        "battery_soc_pct": round(reading["battery_soc"], 1),
    }

    what_if_ctx = None
    if question_type == "explain_what_if":
        what_if_ctx = simulate_what_if(
            station.id, state["last_scenario"], state["last_index"],
            solar_capacity_change_pct=20, battery_capacity_change_pct=50,
        )

    context = AssistantContext(
        station_id=station.id, station_name=station.name, energy_type=station.energy_type,
        scenario=state["last_scenario"], current_index=state["last_index"],
        timestamp=reading["timestamp"], current_state=current_state,
        forecast=state["last_forecast"] if question_type == "explain_forecast" else None,
        decision=state["last_result"] if question_type in ("explain_current_status", "explain_decision") else None,
        stations_overview=_assistant_stations_overview() if question_type == "compare_stations" else None,
        what_if=what_if_ctx,
    )
    result = answer_question(question, context)

    facts_html = "".join(
        f'<div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;padding:8px 12px;min-width:110px;">'
        f'<div style="font-size:10px;text-transform:uppercase;color:#64748b;">{f["label"]}</div>'
        f'<div style="font-size:14px;font-weight:600;color:#0f172a;">{f["value"]}{(" " + f["unit"]) if f.get("unit") else ""}</div></div>'
        for f in result["facts"]
    )
    display(Markdown(f"**Q: {question}**"))
    display(HTML(
        f'<div style="background:#eef2ff;border:1px solid #c7d2fe;border-radius:10px;padding:12px;'
        f'font-size:14px;color:#1e293b;">{result["answer"]}</div>'
    ))
    if facts_html:
        display(HTML(f'<div style="display:flex;flex-wrap:wrap;gap:8px;margin-top:8px;">{facts_html}</div>'))
    display(HTML(
        f'<div style="font-size:11px;color:#94a3b8;margin-top:6px;">Intent: {result["intent"]} · '
        f'Station: {result["station_id"]} · Scenario: {result["grounding"]["scenario"]} · '
        f'Mode: {result["grounding"]["mode"]}</div>'
    ))


assistant_view = widgets.interactive_output(run_assistant, {"question_type": assistant_dd})
display(assistant_dd, assistant_view)


## 8. KPI Summary

The web dashboard doesn't currently expose an aggregate KPI view — it only shows per-decision numbers on the Decision Card and History timeline. This section aggregates those *same* per-decision fields (`expected_kwh`, `expected_value_egp`, `co2_avoided_kg`, and, since Part 3, `expected_cost_egp`/`co2_emitted_kg` for deficit actions) across everything you've logged in this notebook session via **Log this Decision** above, so nothing here is a new or independently-derived number — it's a sum over numbers `ra_core.decision_engine` already produced. Renewable Utilization/Curtailment Avoided only count *surplus*-mode decisions (their original meaning); Battery Discharged/Grid Import/CO2 Emitted are new, deficit-side totals.

In [ ]:
kpi_out = widgets.Output()
kpi_refresh = widgets.Button(description="Recalculate KPI Summary", button_style="info")


def render_kpi(_=None):
    scenario = state["last_scenario"]
    df = state["dfs"][scenario]
    all_rows = df.to_dict("records")
    available = kpi.total_available_surplus_kwh(all_rows, DEFAULT_START_INDEX, state["last_index"])
    session_decisions = [d for d in state["decision_log"] if d["scenario"] == scenario]
    summary = kpi.summarize(session_decisions, available)
    with kpi_out:
        clear_output(wait=True)
        cards = [
            _card("Decisions Logged", summary["decisions_logged"]),
            _card("Renewable Utilization", f'{summary["renewable_utilization_pct"]:.1f}%'),
            _card("Curtailment Avoided", f'{summary["curtailment_avoided_kwh"]:.1f} kWh'),
            _card("Total Value", f'{summary["total_value_egp"]:.1f} EGP'),
            _card("CO2 Avoided", f'{summary["total_co2_avoided_kg"]:.1f} kg'),
            _card("Battery Discharged", f'{summary["battery_discharge_kwh"]:.1f} kWh'),
            _card("Grid Import", f'{summary["grid_import_kwh"]:.1f} kWh', f'{summary["grid_import_cost_egp"]:.1f} EGP'),
            _card("CO2 Emitted", f'{summary["co2_emitted_kg"]:.1f} kg'),
        ]
        display(HTML(f'<div style="display:flex;flex-wrap:wrap;gap:10px;">{"".join(cards)}</div>'))
        display(HTML(
            f'<div style="font-size:12px;color:#94a3b8;margin-top:6px;">Window: {scenario} scenario, '
            f'step {DEFAULT_START_INDEX} → {state["last_index"]} '
            f'({available:.1f} kWh of surplus was available in that window).</div>'
        ))


kpi_refresh.on_click(render_kpi)
display(kpi_refresh, kpi_out)
render_kpi()


## 9. History Log

Every decision you've clicked **Log this Decision** on, across scenarios, in this notebook session.

In [ ]:
history_out = widgets.Output()
history_refresh = widgets.Button(description="Refresh History Log")


def render_history(_=None):
    with history_out:
        clear_output(wait=True)
        if not state["decision_log"]:
            display(HTML(
                '<div style="color:#94a3b8;font-size:13px;">No decisions logged yet — '
                'use "Log this Decision" in Section 2–5 above.</div>'
            ))
            return
        def _row(d):
            is_grid_import = d["action"] == "grid_import"
            money = -d.get("expected_cost_egp", 0.0) if is_grid_import else d["expected_value_egp"]
            co2 = d.get("co2_emitted_kg", 0.0) if is_grid_import else d["co2_avoided_kg"]
            money_color = "#e11d48" if money < 0 else "#059669"
            mode = d.get("mode", "surplus")
            return (
                f'<tr><td style="padding:6px 10px;">{d["timestamp"].replace("T", " ")[:16]}</td>'
                f'<td style="padding:6px 10px;">{d["scenario"]}</td>'
                f'<td style="padding:6px 10px;text-transform:capitalize;">{mode}</td>'
                f'<td style="padding:6px 10px;">{ACTION_LABELS.get(d["action"], d["action"])}</td>'
                f'<td style="padding:6px 10px;text-align:right;color:{money_color};">{money:.1f} EGP</td>'
                f'<td style="padding:6px 10px;text-align:right;color:#0284c7;">{co2:.1f} kg</td></tr>'
            )
        rows = "".join(_row(d) for d in reversed(state["decision_log"]))
        display(HTML(
            '<table style="border-collapse:collapse;width:100%;font-size:13px;">'
            '<thead><tr style="border-bottom:2px solid #e2e8f0;color:#64748b;text-align:left;">'
            '<th style="padding:6px 10px;">Time</th><th>Scenario</th><th>Mode</th><th>Action</th>'
            '<th style="text-align:right;">Value/Cost</th><th style="text-align:right;">CO2</th>'
            f'</tr></thead><tbody>{rows}</tbody></table>'
        ))


history_refresh.on_click(render_history)
display(history_refresh, history_out)
render_history()


## Running as a standalone dashboard (optional)

Everything above works from **Run All** inside Jupyter/JupyterLab as-is. If you want a more polished, code-free presentation for a demo, this same notebook can be rendered as a standalone page with [Voilà](https://voila.readthedocs.io/):

```bash
pip install voila
voila RA_notebook_demo.ipynb
```

This hides all code cells and shows only the widgets/outputs. It's optional — the notebook works fine on its own.